# Step 1
Download reference genome

In [4]:
!rsync -a -P rsync://hgdownload.soe.ucsc.edu/goldenPath/hg38/chromosomes/chr10.fa.gz ./

receiving incremental file list
chr10.fa.gz
     43,157,332 100%   50.01MB/s    0:00:00 (xfr#1, to-chk=0/1)


Decompress

In [102]:
!gunzip chr10.fa.gz

Download samples

In [103]:
!curl -L -o illumina.fq.bz2 https://github.com/inumanag/fall25-csc-bioinf/raw/refs/heads/main/week4/data/illumina.fq.bz2
!curl -L -o pacbio.fq.bz2 https://github.com/inumanag/fall25-csc-bioinf/raw/refs/heads/main/week4/data/pacbio.fq.bz2

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 20.7M  100 20.7M    0     0  16.3M      0  0:00:01  0:00:01 --:--:-- 16.3M
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 3253k  100 3253k    0     0  4180k      0 --:--:-- --:--:-- --:--:-- 4180k


Decompress

In [104]:
!bzip2 -dkf illumina.fq.bz2
!bzip2 -dkf pacbio.fq.bz2

# Step 2
Align samples to reference genome

Install minimap2

In [105]:
!curl -L  https://github.com/lh3/minimap2/releases/download/v2.30/minimap2-2.30_x64-linux.tar.bz2 -o minimap2.tar.bz2
!tar -jxvf minimap2.tar.bz2

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 2189k  100 2189k    0     0  5100k      0 --:--:-- --:--:-- --:--:-- 5100k
minimap2-2.30_x64-linux/
minimap2-2.30_x64-linux/cookbook.md
minimap2-2.30_x64-linux/paftools.js
minimap2-2.30_x64-linux/NEWS.md
minimap2-2.30_x64-linux/README.md
minimap2-2.30_x64-linux/k8
minimap2-2.30_x64-linux/FAQ.md
minimap2-2.30_x64-linux/minimap2.1
minimap2-2.30_x64-linux/LICENSE.txt
minimap2-2.30_x64-linux/minimap2
minimap2-2.30_x64-linux/README-js.md


Align with minimap2

In [ ]:
!minimap2-2.30_x64-linux/minimap2 -ax map-ont chr10.fa pacbio.fq | samtools view -bS -o sample_pacbio.bam
!samtools sort -o sample_pacbio.bam sample_pacbio.bam
!samtools index sample_pacbio.bam

[ERROR] failed to open file 'chr10.fa.gz': No such file or directory
[main_samview] fail to read the header from "-".


In [ ]:
!minimap2-2.30_x64-linux/minimap2 -ax sr chr10.fa illumina.fq | samtools view -bS -o sample_illumina.bam
!samtools sort -o sample_illumina.bam sample_illumina.bam
!samtools index sample_illumina.bam

[ERROR] failed to open file 'chr10.fa.gz': No such file or directory
[main_samview] fail to read the header from "-".


# Step 3
Call variants

Install freebayes

In [108]:
!curl -L https://github.com/freebayes/freebayes/releases/download/v1.3.10/freebayes-1.3.10-linux-amd64-static.gz -o freebayes.gz
!gunzip --force freebayes.gz
!chmod +x freebayes

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 2436k  100 2436k    0     0  5979k      0 --:--:-- --:--:-- --:--:-- 5979k


Use freebayes

In [110]:
!./freebayes -f chr10.fa sample_pacbio.bam > variants_pacbio.vcf
!./freebayes -f chr10.fa sample_illumina.bam > variants_illumina.vcf

index file chr10.fa.fai not found, generating...
